# Registro histórico: 06_efficientnet_f1

Origem: `C65` — `Colab Notebooks/notebook_otimizado.ipynb`.

EfficientNet-B0 com peso positivo e seleção por F1 de melanoma.

**Limitações conhecidas:** Caminhos conflitantes no mesmo notebook; resumo diz 50 épocas, log para antes. Resultado histórico não reproduzido.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
import os
import random
import shutil
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import transforms, datasets
from torchvision.models import EfficientNet_B0_Weights
from sklearn.metrics import f1_score
import warnings

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# Set random seeds for reproducibility
random.seed(91432)
np.random.seed(91432)
torch.manual_seed(91432)
if torch.cuda.is_available():
    torch.cuda.manual_seed(91432)


In [ ]:
if not os.path.exists("/content/balanced_isic.zip"):
    print("Importando dataset...")
    from google.colab import drive
    drive.mount('/content/drive')

    # Copy dataset from Drive to Colab
    !cp -r "/content/drive/MyDrive/balanced_isic.zip" "/content/balanced_ham10000"
else:
    print("Dataset já importado")



In [ ]:
!unzip /content/balanced_ham10000 -d /content/balanced_ham10000-folder/

In [ ]:
!unzip -l content/balanced_isic.zip

In [ ]:
# Setup paths
image_path = Path("/content/balanced_ham10000-folder/content/ham10000_balanced_images")
train_dir = image_path / "train"
test_dir = image_path / "test"
val_dir = image_path / "val"

print(f"Dataset path: {image_path}")
print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")
print(f"Validation directory: {val_dir}")

# 2. Importando o Dataset



In [ ]:
# # if not os.path.exists("/content/0-vgg"):
# #     print("Importando dataset...")
# #     from google.colab import drive
# #     drive.mount('/content/drive')

# #     # Copy dataset from Drive to Colab
# #     !cp -r "/content/drive/MyDrive/0-vgg" "/content/"
# # else:
# #     print("Dataset já importado")


# # Setup paths
# image_path = Path("/content/0-vgg/base_dir")
# train_dir = image_path / "train_dir"
# test_dir = image_path / "test_dir"
# val_dir = image_path / "val_dir"

# print(f"Dataset path: {image_path}")
# print(f"Train directory: {train_dir}")
# print(f"Test directory: {test_dir}")
# print(f"Validation directory: {val_dir}")


In [ ]:
# Setup paths
image_path = Path("/content/0-vgg/base_dir")
train_dir = image_path / "train_dir"
test_dir = image_path / "test_dir"
val_dir = image_path / "val_dir"

print(f"Dataset path: {image_path}")
print(f"Train directory: {train_dir}")
print(f"Test directory: {test_dir}")
print(f"Validation directory: {val_dir}")


# 3. Download Helper Functions


In [ ]:
# Download helper functions from Learn PyTorch repo
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping download")
else:
    print("Downloading helper_functions.py")
    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
    with open("helper_functions.py", "wb") as f:
        f.write(request.content)

from helper_functions import accuracy_fn
import torch
from sklearn.metrics import f1_score

def f1_score_melanoma(y_true: torch.Tensor, y_pred: torch.Tensor, class_names: list) -> float:
    """
    Calcula o F1-Score para a classe 'melanoma'.
    Assume que a classe 'melanoma' é a segunda classe (índice 1) na lista class_names.
    """
    # Converte para numpy para usar sklearn
    y_true_np = y_true.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()

    # O f1_score do sklearn pode calcular para múltiplas classes.
    # Se for classificação binária, o índice 1 é a classe positiva (melanoma).
    # Se for multiclasse, precisamos garantir que 'melanoma' seja o alvo.

    # Para classificação binária (0 ou 1), o f1_score com average='binary'
    # calcula para a classe positiva (1).

    # Se o código original for binário, y_true e y_pred são 0 ou 1.
    # Vamos assumir que a classe positiva (1) é 'melanoma' para o cálculo do F1.

    # Se houver mais de 2 classes, o usuário precisará ajustar o mapeamento.
    # No contexto de classificação binária (melanoma vs não-melanoma),
    # a classe 1 (melanoma) é a classe de interesse.

    # Se a lista de classes for ['nao_melanoma', 'melanoma'], o índice 1 é melanoma.

    # Para garantir, vamos usar o f1_score com average='binary' e pos_label=1.
    # Isso funciona para o caso binário onde 1 é a classe de interesse.

    try:
        f1 = f1_score(y_true_np, y_pred_np, average='binary', pos_label=1)
    except ValueError:
        # Caso especial onde y_true ou y_pred não contêm a classe 1,
        # o que pode acontecer em batches, mas não deve acontecer no conjunto de validação total.
        # Se acontecer, retornamos 0.0 para evitar quebra.
        f1 = 0.0

    return f1

class EarlyStoppingF1:
    """
    Early stops the training if validation F1-Score for 'melanoma' doesn't improve after a given patience.
    """
    def __init__(self, patience=10, verbose=False, delta=0, path='best_model_f1.pth', class_names=None):
        """
        Args:
            patience (int): How long to wait after last time validation F1-Score improved.
                            Default: 10
            verbose (bool): If True, prints a message for each F1-Score improvement.
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                           Default: 0
            path (str): Path for the checkpoint to be saved to.
                        Default: 'best_model_f1.pth'
            class_names (list): List of class names to determine the index of 'melanoma'.
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_f1_max = -float('inf')
        self.delta = delta
        self.path = path
        self.class_names = class_names

        # Determinar o índice da classe 'melanoma'
        if self.class_names and 'melanoma' in self.class_names:
            self.melanoma_index = self.class_names.index('melanoma')
        else:
            # Assumir que a classe positiva (1) é melanoma no caso binário
            self.melanoma_index = 1

    def __call__(self, val_f1, model):
        score = val_f1

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_f1, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_f1, model)
            self.counter = 0

    def save_checkpoint(self, val_f1, model):
        '''Saves model when validation F1-Score for melanoma improves.'''
        if self.verbose:
            print(f'Validation F1-Score improved ({self.val_f1_max:.6f} --> {val_f1:.6f}). Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_f1_max = val_f1

# Função para calcular todas as métricas de validação
def calculate_val_metrics(model: torch.nn.Module,
                          data_loader: torch.utils.data.DataLoader,
                          loss_fn: torch.nn.Module,
                          device: torch.device,
                          class_names: list):
    """Calcula perda, acurácia e F1-Score para melanoma no conjunto de validação."""
    model.eval()
    val_loss, val_acc = 0, 0
    all_preds = []
    all_labels = []

    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            y_float = y.float().unsqueeze(1)

            logits = model(X)
            loss = loss_fn(logits, y_float)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()

            acc = accuracy_fn(y_true=y, y_pred=preds.squeeze(1))

            val_loss += loss.item()
            val_acc += acc

            all_preds.extend(preds.squeeze(1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    # Converte listas para tensores para o cálculo do F1-Score
    all_labels_tensor = torch.tensor(all_labels)
    all_preds_tensor = torch.tensor(all_preds)

    # Calcula F1-Score para melanoma
    val_f1 = f1_score_melanoma(all_labels_tensor, all_preds_tensor, class_names)

    return val_loss, val_acc, val_f1


# 4. Visualização e levantamento do dataset


In [ ]:
def count_images_per_class(directory):
    """Counts the number of image files per class in a directory."""
    class_counts = {}
    for class_name in os.listdir(directory):
        class_dir = os.path.join(directory, class_name)
        if os.path.isdir(class_dir):
            image_count = 0
            for _, _, files in os.walk(class_dir):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                        image_count += 1
            class_counts[class_name] = image_count
    return class_counts

# Count images per class in each directory
train_class_counts = count_images_per_class(train_dir)
test_class_counts = count_images_per_class(test_dir)
val_class_counts = count_images_per_class(val_dir)

print("Number of images per class:")
print(f"  Train directory: {train_class_counts}")
print(f"  Test directory: {test_class_counts}")
print(f"  Validation directory: {val_class_counts}")

# Get all image paths
image_path_list = list(image_path.glob("*/*/*.jpg"))
print(f"\nTotal images found: {len(image_path_list)}")

# Display a random image
random_image_path = random.choice(image_path_list)
image_class = random_image_path.parent.stem
print(f"\nRandom image path: {random_image_path}")
print(f"Image class: {image_class}")

img = Image.open(random_image_path)
print(f"Image dimensions: {img.width}x{img.height}")
print(f"Image mode: {img.mode}")

plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title(f"Sample Image - Class: {image_class}", fontsize=14)
plt.axis("off")
plt.show()


# 5. Data transforms para a arquitetura EfficientNet B0


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to slightly larger first
    transforms.RandomCrop(224),     # Random crop for data augmentation

    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                        std=[0.229, 0.224, 0.225])
])

# Test/validation transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Data transforms created successfully!")


# 6. Visualização das imagens Transformadas


In [ ]:
def plot_transformed_images(image_paths, transform, n=3, seed=42):
    """Plot original and transformed images side by side."""
    if seed:
        random.seed(seed)
    random_image_paths = random.sample(image_paths, k=n)

    fig, axes = plt.subplots(n, 2, figsize=(10, 4*n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for i, image_path in enumerate(random_image_paths):
        with Image.open(image_path) as f:
            # Original image
            axes[i, 0].imshow(f)
            axes[i, 0].set_title("Original")
            axes[i, 0].axis("off")

            # Transformed image
            transformed_image = transform(f).permute(1, 2, 0)
            # Denormalize for visualization
            mean = torch.tensor([0.485, 0.456, 0.406])
            std = torch.tensor([0.229, 0.224, 0.225])
            transformed_image = transformed_image * std + mean
            transformed_image = torch.clamp(transformed_image, 0, 1)

            axes[i, 1].imshow(transformed_image)
            axes[i, 1].set_title("Transformed")
            axes[i, 1].axis("off")

            fig.suptitle(f"Class: {image_path.parent.stem}", fontsize=16)

    plt.tight_layout()
    plt.show()

plot_transformed_images(image_paths=image_path_list,
                       transform=train_transform,
                       n=2,
                       seed=42)


# 7. Criando os Dataloaders


In [ ]:
# Create datasets
train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_data = datasets.ImageFolder(root=test_dir, transform=test_transform)
val_data = datasets.ImageFolder(root=val_dir, transform=test_transform)

# Get class information
class_names = train_data.classes
class_dict = train_data.class_to_idx
print(f"Classes: {class_names}")
print(f"Class to index mapping: {class_dict}")
print(f"Dataset sizes - Train: {len(train_data)}, Test: {len(test_data)}, Val: {len(val_data)}")

BATCH_SIZE = 32  # Smaller batch size for better convergence

# Create dataloaders
train_dataloader = DataLoader(dataset=train_data,
                             batch_size=BATCH_SIZE,
                             shuffle=True,
                             num_workers=0,  # For faster data loading
                             pin_memory=True if device == "cuda" else False)

val_dataloader = DataLoader(dataset=val_data,
                            batch_size=BATCH_SIZE,
                            shuffle=False,
                            num_workers=0,
                            pin_memory=True if device == "cuda" else False)

test_dataloader = DataLoader(dataset=test_data,
                           batch_size=BATCH_SIZE,
                           shuffle=False,
                           num_workers=0,
                           pin_memory=True if device == "cuda" else False)

print(f"DataLoaders created - Train batches: {len(train_dataloader)}, Test batches: {len(test_dataloader)}, Val batches: {len(val_dataloader)}")


# 8. Criando o Modelo EfficientNet B0


In [ ]:
def create_efficientnet_b0_model(num_classes=1, pretrained=True):
    """Create and modify EfficientNet B0 for binary classification."""
    if pretrained:
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        print("Loaded EfficientNet_B0 with ImageNet pretrained weights")
    else:
        model = models.efficientnet_b0(weights=None)
        print("Created EfficientNet_B0 without pretrained weights")

    # (opcional) congelar camadas iniciais para transfer learning
    for param in model.features[:-3].parameters():
        param.requires_grad = False

    # Modificar o classifier para 1 saída (logit)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)

    print(f"Modified classifier for {num_classes} class")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    return model

# Create model
model = create_efficientnet_b0_model(num_classes=1, pretrained=True)
model = model.to(device)


# 9. Setup do treinamento


In [ ]:
"""# 9. Setup do treinamento (otimizado para BINÁRIO + desbalanceamento)"""

# Calcula a proporção de classes no treino para usar em pos_weight
# Atenção: assumimos que a classe 1 é MELANOMA (positiva)
train_targets = np.array(train_data.targets)
class_counts = np.bincount(train_targets)
print(f"Contagem de classes no treino: {class_counts}")

if len(class_counts) == 2:
    num_negativos = class_counts[0]
    num_positivos = class_counts[1]
    # pos_weight > 1 penaliza erros na classe positiva (melanoma)
    pos_weight_value = num_negativos / max(num_positivos, 1)
    pos_weight = torch.tensor([pos_weight_value], device=device)
    print(f"Usando pos_weight para BCEWithLogitsLoss: {pos_weight_value:.4f}")
else:
    # fallback se por algum motivo não for binário
    pos_weight = torch.tensor([1.0], device=device)
    print("Aviso: dataset não parece binário, usando pos_weight = 1.0")

# Loss function otimizada para classificação binária desbalanceada
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Otimizador AdamW com weight decay para regularização (evitar overfitting)
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,       # LR baixa para fine-tuning
    weight_decay=1e-4
)

# Scheduler para reduzir LR quando a validação "empaca"
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

print("Training configuration setup complete!")
print(f"Loss function: {loss_fn}")
print(f"Optimizer: {optimizer}")
print(f"Learning rate scheduler: {scheduler}")


# 10. Funções de treino e validação


In [ ]:
"""# 10. Funções de treino e validação (com F1 na validação)"""

def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device = device):
    model.train()
    train_loss, train_acc = 0, 0

    progress_bar = tqdm(data_loader, desc="Training", leave=False)

    for batch, (X, y) in enumerate(progress_bar):
        # Envia para device
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

        # y: [B] -> [B,1] e float pra BCE
        y_float = y.float().unsqueeze(1)          # [B,1]

        # Forward: logits [B,1]
        logits = model(X)

        # Loss com logits
        loss = loss_fn(logits, y_float)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Probabilidades e predições binárias
        probs = torch.sigmoid(logits)            # [B,1]
        preds = (probs > 0.5).long()             # [B,1] 0/1

        # accuracy_fn espera [B] com ints
        acc = accuracy_fn(
            y_true=y,                            # [B]
            y_pred=preds.squeeze(1)              # [B]
        )

        # Acumula métricas
        train_loss += loss.item()
        train_acc += acc

        progress_bar.set_postfix({
            'batch_loss': f'{loss.item():.4f}',
            'batch_acc': f'{acc:.2f}%'
        })

    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_acc


def val_step(model: torch.nn.Module,
             data_loader: torch.utils.data.DataLoader,
             loss_fn: torch.nn.Module,
             device: torch.device = device):
    """
    Passo de validação retornando loss, acc e F1 para a classe positiva (melanoma = 1).
    """
    model.eval()
    val_loss, val_acc = 0, 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(data_loader, desc="Validation", leave=False)

    with torch.inference_mode():
        for batch, (X, y) in enumerate(progress_bar):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

            y_float = y.float().unsqueeze(1)     # [B,1]
            logits = model(X)                    # [B,1]
            loss = loss_fn(logits, y_float)

            probs = torch.sigmoid(logits)        # [B,1]
            preds = (probs > 0.5).long()         # [B,1]

            acc = accuracy_fn(
                y_true=y,
                y_pred=preds.squeeze(1)
            )

            val_loss += loss.item()
            val_acc += acc

            all_labels.extend(y.cpu().tolist())
            all_preds.extend(preds.squeeze(1).cpu().tolist())

            progress_bar.set_postfix({
                'batch_loss': f'{loss.item():.4f}',
                'batch_acc': f'{acc:.2f}%'
            })

    val_loss /= len(data_loader)
    val_acc /= len(data_loader)

    # F1 para classe positiva (assumindo melanoma = 1)
    val_f1 = f1_score(all_labels, all_preds, pos_label=1)

    return val_loss, val_acc, val_f1


# 11. Função de Treinamento Principal


In [ ]:
"""# 11. Função de Treinamento Principal (com Early Stopping por F1)"""

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          val_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
          epochs: int,
          device: torch.device = device,
          patience: int = 10,
          min_delta: float = 1e-4,
          best_model_path: str = "best_model_f1.pth"):
    """
    Treina e valida o modelo, com early stopping baseado em F1 (classe melanoma = 1).

    - Para quando 'patience' épocas passarem sem aumento significativo de F1.
    - Salva o melhor modelo (maior F1) em best_model_path.
    """

    # Dicionário de resultados
    results = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": []
    }

    best_val_f1 = 0.0
    best_val_acc = 0.0
    epochs_sem_melhora = 0

    for epoch in tqdm(range(epochs), desc="Epochs"):
        train_loss, train_acc = train_step(
            model=model,
            data_loader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )

        val_loss, val_acc, val_f1 = val_step(
            model=model,
            data_loader=val_dataloader,
            loss_fn=loss_fn,
            device=device
        )

        # Atualiza dicionário de resultados
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)
        results["val_f1"].append(val_f1)

        # Print por época
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.2f}% | "
            f"val_loss: {val_loss:.4f} | "
            f"val_acc: {val_acc:.2f}% | "
            f"val_f1 (melanoma): {val_f1:.4f}"
        )

        # Scheduler baseado em loss de validação
        scheduler.step(val_loss)

        # Early Stopping baseado em F1 (classe positiva)
        if val_f1 - best_val_f1 > min_delta:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            epochs_sem_melhora = 0

            # Salva melhor modelo (em termos de F1)
            torch.save(model.state_dict(), best_model_path)
            print(f"⬆️ Novo melhor F1 de validação: {best_val_f1:.4f} | modelo salvo em {best_model_path}")
        else:
            epochs_sem_melhora += 1
            print(f"Sem melhora significativa em F1 há {epochs_sem_melhora} época(s).")

            if epochs_sem_melhora >= patience:
                print(f"⏹️ Early stopping acionado na época {epoch+1} (paciente = {patience}).")
                break

    print(f"Melhor F1 de validação (melanoma): {best_val_f1:.4f}")
    print(f"Melhor acurácia de validação correspondente: {best_val_acc:.2f}%")

    return results, best_val_acc, best_val_f1


# 12. Treinamento do Modelo


In [ ]:
"""# 12. Treinamento do Modelo (com Early Stopping)"""

# Número máximo de épocas (early stopping pode parar antes)
EPOCHS = 50  # por exemplo, pode deixar maior já que o early stopping controla

BEST_MODEL_PATH = "best_model_f1.pth"

# Timer
start_time = timer()

# Treino
model_results, best_val_acc, best_val_f1 = train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    scheduler=scheduler,
    epochs=EPOCHS,
    device=device,
    patience=10,          # para após 10 épocas sem melhora em F1
    min_delta=1e-4,
    best_model_path=BEST_MODEL_PATH
)

end_time = timer()
total_train_time = end_time - start_time

print(f"Total training time: {total_train_time:.2f} seconds")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Best Validation F1 (melanoma): {best_val_f1:.4f}")

# Opcional: recarregar o melhor modelo antes de testar
print(f"Carregando o melhor modelo salvo em {BEST_MODEL_PATH} para avaliação final...")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.to(device)


# 13. Resultados do Teste


In [ ]:
"""# 13. Resultados do Teste (corrigido para BCEWithLogitsLoss)"""

def test_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               device: torch.device = device):
    """Tests a model on a test dataset."""
    model.eval()
    test_loss, test_acc = 0, 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(data_loader, desc="Testing", leave=False)

    with torch.inference_mode():
        for X, y in progress_bar:
            X, y = X.to(device), y.to(device)

            # 1. Forward pass -> logits [B,1]
            y_logits = model(X)

            # 2. Calcula loss em LOGITS
            y_float = y.float().unsqueeze(1)          # [B,1]
            loss = loss_fn(y_logits, y_float)
            test_loss += loss.item()

            # 3. Probabilidades e classe prevista
            y_pred_prob = torch.sigmoid(y_logits).squeeze(1)   # [B]
            y_pred_class = (y_pred_prob > 0.5).long()          # [B]

            # Acurácia
            test_acc += accuracy_fn(
                y_true=y.cpu(),
                y_pred=y_pred_class.cpu()
            )

            # Guarda para métricas finais
            all_preds.extend(y_pred_class.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    test_loss = test_loss / len(data_loader)
    test_acc = test_acc / len(data_loader)

    return {
        "model_name": model.__class__.__name__,
        "model_loss": test_loss,
        "model_acc": test_acc,
        "predictions": all_preds,
        "labels": all_labels
    }

# Teste
test_results = test_model(
    model=model,
    data_loader=test_dataloader,
    loss_fn=loss_fn,
    device=device
)

print(f"Test Loss: {test_results['model_loss']:.4f}")
print(f"Test Accuracy: {test_results['model_acc']:.2f}%")


# 14. Salvando o Modelo e Histórico de Treinamento


In [ ]:
import json

# Define paths for saving
LOCAL_MODEL_PATH = Path("models")
DRIVE_MODEL_PATH = Path("/content/drive/MyDrive/Trained_Models")

LOCAL_MODEL_PATH.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist
DRIVE_MODEL_PATH.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist

# O melhor modelo já foi salvo pelo Early Stopping como 'best_model_f1_melanoma.pth'
# e carregado de volta para o objeto 'model'.
# Agora, vamos renomear e salvar o modelo final (o melhor) no Drive.

MODEL_NAME = "EfficientNetB0_SkinLesion_Binary_Classifier_BEST_F1.pth"

# --- Logic for local model saving ---
MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / MODEL_NAME

# Check if local model file already exists and append a number if necessary
counter = 1
original_model_name_local = MODEL_SAVE_PATH_LOCAL.stem
while MODEL_SAVE_PATH_LOCAL.exists():
    MODEL_SAVE_PATH_LOCAL = LOCAL_MODEL_PATH / f"{original_model_name_local}_{counter}{MODEL_SAVE_PATH_LOCAL.suffix}"
    counter += 1

print(f"Saving final best model locally to: {MODEL_SAVE_PATH_LOCAL}")
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH_LOCAL)

# --- Logic for Google Drive model saving ---
MODEL_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / MODEL_NAME

# Check if Drive model file already exists and append a number if necessary
counter = 1
original_model_name_drive = MODEL_SAVE_PATH_DRIVE.stem
while MODEL_SAVE_PATH_DRIVE.exists():
    MODEL_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / f"{original_model_name_drive}_{counter}{MODEL_SAVE_PATH_DRIVE.suffix}"
    counter += 1

print(f"Saving final best model to Google Drive: {MODEL_SAVE_PATH_DRIVE}")
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH_DRIVE)

# Save training history
history = {
    "train_losses": model_results["train_loss"],
    "val_losses": model_results["val_loss"],
    "train_accs": model_results["train_acc"],
    "val_accs": model_results["val_acc"],
    "test_results": {
        "loss": test_results['model_loss'],
        "accuracy": test_results['model_acc']
    },
    "training_time": total_train_time,
    "epochs": EPOCHS,
    "best_val_f1": best_val_f1
}

# Define the path for the training history JSON in Google Drive
HISTORY_NAME = "training_history_EfficientNetB0_1.json"
HISTORY_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / HISTORY_NAME

# Check if history file already exists and append a number if necessary
counter = 1
original_history_name = HISTORY_SAVE_PATH_DRIVE.stem
while HISTORY_SAVE_PATH_DRIVE.exists():
    HISTORY_SAVE_PATH_DRIVE = DRIVE_MODEL_PATH / f"{original_history_name}_{counter}{HISTORY_SAVE_PATH_DRIVE.suffix}"
    counter += 1

with open(HISTORY_SAVE_PATH_DRIVE, "w") as f:
    json.dump(history, f, indent=2)

print("Model and training history saved successfully to Google Drive!")


# 15. Fazendo Previsões


In [ ]:
"""# 15. Fazendo Previsões (binário)"""

def make_predictions(model: torch.nn.Module,
                     data: list,
                     device: torch.device = device):
    """
    Faz previsões em uma lista de tensores de imagem.

    Retorna um tensor [N] com a probabilidade da classe positiva (melanoma = 1).
    """
    pred_probs = []
    model.eval()

    with torch.inference_mode():
        for sample in data:
            sample = torch.unsqueeze(sample, dim=0).to(device)   # [1,C,H,W]

            logits = model(sample)                               # [1,1]
            prob_pos = torch.sigmoid(logits).squeeze().cpu()     # escalar

            pred_probs.append(prob_pos)

    return torch.stack(pred_probs)   # [N]


def visualize_predictions(model, dataset, class_names, num_samples=6):
    """Visualiza previsões do modelo em amostras aleatórias (binário)."""
    # Amostras aleatórias
    test_samples = []
    test_labels = []

    for sample, label in random.sample(list(dataset), k=num_samples):
        test_samples.append(sample)
        test_labels.append(label)

    # Probabilidades da classe positiva
    pred_probs_pos = make_predictions(model=model, data=test_samples)  # [N]

    # Constrói probabilidades das duas classes: [N,2]
    probs_neg = 1 - pred_probs_pos
    probs_pos = pred_probs_pos
    two_class_probs = torch.stack([probs_neg, probs_pos], dim=1)       # [N,2]

    pred_classes = two_class_probs.argmax(dim=1)                       # [N]

    # Visualização
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()

    for i, sample in enumerate(test_samples):
        # Desnormaliza imagem para exibição
        img = sample.permute(1, 2, 0)
        mean = torch.tensor([0.485, 0.456, 0.406])
        std = torch.tensor([0.229, 0.224, 0.225])
        img = img * std + mean
        img = torch.clamp(img, 0, 1)

        axes[i].imshow(img)

        pred_idx = int(pred_classes[i])
        truth_idx = int(test_labels[i])

        pred_label = class_names[pred_idx]
        truth_label = class_names[truth_idx]

        confidence = two_class_probs[i, pred_idx].item() * 100

        title_text = f"Pred: {pred_label} ({confidence:.1f}%)\nTruth: {truth_label}"

        if pred_idx == truth_idx:
            axes[i].set_title(title_text, fontsize=12, color="green", fontweight='bold')
        else:
            axes[i].set_title(title_text, fontsize=12, color="red", fontweight='bold')

        axes[i].axis("off")

    plt.tight_layout()
    plt.suptitle("Model Predictions on Validation Samples (Binary)", fontsize=16, fontweight='bold', y=1.02)
    plt.show()


# Visualizar previsões
print("Visualizing model predictions (binary)...")
visualize_predictions(model, val_data, class_names, num_samples=6)


# 16. Sumário e Resultados


In [ ]:
print("=" * 60)
print("📊 TRAINING SUMMARY")
print("=" * 60)
print(f"Model: EfficientNet_B0")
print(f"Task: Binary Classification (Melanoma vs Non-Melanoma)")
print(f"Dataset Size: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")
print(f"Training Time: {total_train_time:.2f} seconds")
print(f"Epochs Trained: {EPOCHS}")
print(f"Best Validation F1-Score (Melanoma): {best_val_f1:.2f}")
print(f"Final Test Accuracy: {test_results['model_acc']:.2f}%")
print("=" * 60)

# Performance metrics
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Generate classification report
y_true = test_results['labels']
y_pred = test_results['predictions']

print("\n📈 Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()


In [ ]:
print("=" * 60)
print("📊 TRAINING SUMMARY")
print("=" * 60)
print(f"Model: EfficientNet_B0")
print(f"Task: Binary Classification (Melanoma vs Non-Melanoma)")
print(f"Dataset Size: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")
print(f"Training Time: {total_train_time:.2f} seconds")
print(f"Epochs Trained: {EPOCHS}")
print(f"Best Validation F1-Score (Melanoma): {best_val_f1:.2f}")
print(f"Final Test Accuracy: {test_results['model_acc']:.2f}%")
print("=" * 60)

# Performance metrics
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Generate classification report
y_true = test_results['labels']
y_pred = test_results['predictions']

print("\n📈 Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Extract data from model_results
train_losses = model_results["train_loss"]
val_losses = model_results["val_loss"]
train_accs = model_results["train_acc"]
val_accs = model_results["val_acc"]
val_f1_scores = model_results["val_f1"]

epochs_ran = len(train_losses)
epochs_range = range(1, epochs_ran + 1)

plt.figure(figsize=(18, 6))

# Plot Loss
plt.subplot(1, 3, 1)
plt.plot(epochs_range, train_losses, label='Train Loss')
plt.plot(epochs_range, val_losses, label='Validation Loss')
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 3, 2)
plt.plot(epochs_range, train_accs, label='Train Accuracy')
plt.plot(epochs_range, val_accs, label='Validation Accuracy')
plt.title('Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

# Plot F1-Score (Melanoma)
plt.subplot(1, 3, 3)
plt.plot(epochs_range, val_f1_scores, label='Validation F1-Score (Melanoma)', color='orange')
plt.title('F1-Score (Melanoma) per Epoch')
plt.xlabel('Epoch')
plt.ylabel('F1-Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()